In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(43)

In [3]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

customers_path = project_root / "data" / "customers.csv"

customers_df = pd.read_csv(customers_path)

customers_df.head(10)

,customer_id,age,region,monthly_income,employment_status
0,C00001,41,Stockholms län,26400,Temporary/Probation/Freelance
1,C00002,68,Stockholms län,18900,Retired
2,C00003,56,Dalarnas län,30800,Permanent employment
3,C00004,49,Västra Götalands län,35700,Permanent employment
4,C00005,27,Östergötlands län,45900,Permanent employment
5,C00006,33,Skåne län,29900,Self-employed
6,C00007,20,Stockholms län,57100,Self-employed
7,C00008,67,Västra Götalands län,24000,Retired
8,C00009,48,Stockholms län,42600,Permanent employment
9,C00010,63,Västra Götalands län,40000,Permanent employment


In [4]:
print("Antal kunder:", len(customers_df))
print(customers_df.columns.tolist())

Antal kunder: 10000
['customer_id', 'age', 'region', 'monthly_income', 'employment_status']


In [5]:
# Kodblock 4
loan_counts = []

for age in customers_df["age"]:

    if age <= 24:
        probabilities = [0.88, 0.11, 0.01, 0.00]

    elif age <= 34:
        probabilities = [0.62, 0.30, 0.07, 0.01]

    elif age <= 54:
        probabilities = [0.52, 0.33, 0.12, 0.03]

    elif age <= 66:
        probabilities = [0.60, 0.29, 0.09, 0.02]

    else:
        probabilities = [0.74, 0.22, 0.04, 0.00]

    loan_count = np.random.choice(
        [1, 2, 3, 4],
        p=probabilities
    )

    loan_counts.append(loan_count)

In [6]:
loan_count_series = pd.Series(loan_counts)

print("Antal kunder:", len(loan_counts))

print("\nAntal lån per kund:")
print(loan_count_series.value_counts().sort_index())

print("\nTotalt antal lån:")
print(sum(loan_counts))

Antal kunder: 10000

Antal lån per kund:
1    6129
2    2838
3     858
4     175
Name: count, dtype: int64

Totalt antal lån:
15079


In [7]:
loan_customer_ids = []

for customer_id, loan_count in zip(
    customers_df["customer_id"],
    loan_counts
):
    
    for loan_number in range(loan_count):
        loan_customer_ids.append(customer_id)

In [8]:
print("Antal lånerader:", len(loan_customer_ids))
print(loan_customer_ids[:10])

Antal lånerader: 15079
['C00001', 'C00002', 'C00003', 'C00004', 'C00005', 'C00006', 'C00006', 'C00007', 'C00008', 'C00009']


In [9]:
loan_ids = []

for loan_number in range(1, len(loan_customer_ids) + 1):

    loan_id = f"L{loan_number:06d}"

    loan_ids.append(loan_id)

print(loan_ids[:10])
print("Antal loan IDs:", len(loan_ids))

['L000001', 'L000002', 'L000003', 'L000004', 'L000005', 'L000006', 'L000007', 'L000008', 'L000009', 'L000010']
Antal loan IDs: 15079


In [10]:
loan_types = []

for customer_index, customer in customers_df.iterrows():

    age = customer["age"]
    income = customer["monthly_income"]
    employment_status = customer["employment_status"]

    loan_count = loan_counts[customer_index]

    # Grundsannolikheter
    probabilities = np.array([
        0.40,  # Mortgage
        0.20,  # Personal Loan
        0.15,  # Car Loan
        0.25   # Credit Card
    ])

    # Unga kunder
    if age <= 24:
        probabilities = np.array([
            0.05,
            0.20,
            0.15,
            0.60
        ])

    # Äldre kunder
    elif age >= 67:
        probabilities = np.array([
            0.50,
            0.15,
            0.10,
            0.25
        ])

    # Låg inkomst gör bolån mindre sannolikt
    if income < 20000:
        probabilities[0] *= 0.20
        probabilities[3] *= 1.50

    # Högre inkomst gör bolån mer sannolikt
    elif income >= 45000:
        probabilities[0] *= 1.40

    # Student
    if employment_status == "Student":
        probabilities[0] *= 0.10
        probabilities[3] *= 1.50

    # Arbetslös
    elif employment_status == "Unemployed":
        probabilities[0] *= 0.15
        probabilities[1] *= 0.70
        probabilities[3] *= 1.40

    # Tillsvidareanställd
    elif employment_status == "Permanent employment":
        probabilities[0] *= 1.25

    # Gör om värdena så att de tillsammans blir 100 %
    probabilities = probabilities / probabilities.sum()

    customer_loan_types = np.random.choice(
        ["Mortgage", "Personal Loan", "Car Loan", "Credit Card"],
        size=loan_count,
        replace=False,
        p=probabilities
    )

    loan_types.extend(customer_loan_types)

In [11]:
print("Antal lånetyper:", len(loan_types))

print("\nFördelning:")
print(pd.Series(loan_types).value_counts())

print("\nAndelar:")
print(
    pd.Series(loan_types)
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

Antal lånetyper: 15079

Fördelning:
Mortgage         5084
Credit Card      4486
Personal Loan    3050
Car Loan         2459
Name: count, dtype: int64

Andelar:
Mortgage         33.7
Credit Card      29.7
Personal Loan    20.2
Car Loan         16.3
Name: proportion, dtype: float64


In [12]:
loans_df = pd.DataFrame({
    "loan_id": loan_ids,
    "customer_id": loan_customer_ids,
    "loan_type": loan_types
})

loans_df.head(10)

,loan_id,customer_id,loan_type
0,L000001,C00001,Personal Loan
1,L000002,C00002,Mortgage
2,L000003,C00003,Mortgage
3,L000004,C00004,Car Loan
4,L000005,C00005,Mortgage
5,L000006,C00006,Mortgage
6,L000007,C00006,Car Loan
7,L000008,C00007,Credit Card
8,L000009,C00008,Car Loan
9,L000010,C00009,Mortgage


In [13]:
loans_df = loans_df.merge(
    customers_df[
        [
            "customer_id",
            "age",
            "monthly_income",
            "employment_status"
        ]
    ],
    on="customer_id",
    how="left"
)

loans_df.head(10)

,loan_id,customer_id,loan_type,age,monthly_income,employment_status
0,L000001,C00001,Personal Loan,41,26400,Temporary/Probation/Freelance
1,L000002,C00002,Mortgage,68,18900,Retired
2,L000003,C00003,Mortgage,56,30800,Permanent employment
3,L000004,C00004,Car Loan,49,35700,Permanent employment
4,L000005,C00005,Mortgage,27,45900,Permanent employment
5,L000006,C00006,Mortgage,33,29900,Self-employed
6,L000007,C00006,Car Loan,33,29900,Self-employed
7,L000008,C00007,Credit Card,20,57100,Self-employed
8,L000009,C00008,Car Loan,67,24000,Retired
9,L000010,C00009,Mortgage,48,42600,Permanent employment


In [14]:
original_amounts = []

for index, loan in loans_df.iterrows():

    loan_type = loan["loan_type"]
    monthly_income = loan["monthly_income"]

    annual_income = monthly_income * 12

    # Bolån
    if loan_type == "Mortgage":

        income_multiplier = np.random.uniform(2.0, 4.5)

        original_amount = annual_income * income_multiplier

        original_amount = np.clip(
            original_amount,
            300000,
            6000000
        )

    # Privatlån
    elif loan_type == "Personal Loan":

        income_multiplier = np.random.uniform(1, 8)

        original_amount = monthly_income * income_multiplier

        original_amount = np.clip(
            original_amount,
            10000,
            500000
        )

    # Billån
    elif loan_type == "Car Loan":

        income_multiplier = np.random.uniform(2, 10)

        original_amount = monthly_income * income_multiplier

        original_amount = np.clip(
            original_amount,
            50000,
            700000
        )

    # Kreditkort
    else:

        income_multiplier = np.random.uniform(0.25, 1.5)

        original_amount = monthly_income * income_multiplier

        original_amount = np.clip(
            original_amount,
            5000,
            100000
        )

    # Avrunda till närmaste 1 000 kr
    original_amount = round(original_amount / 1000) * 1000

    original_amounts.append(int(original_amount))

In [15]:
loans_df["original_amount"] = original_amounts

loans_df.head(10)

,loan_id,customer_id,loan_type,age,monthly_income,employment_status,original_amount
0,L000001,C00001,Personal Loan,41,26400,Temporary/Probation/Freelance,186000
1,L000002,C00002,Mortgage,68,18900,Retired,470000
2,L000003,C00003,Mortgage,56,30800,Permanent employment,917000
3,L000004,C00004,Car Loan,49,35700,Permanent employment,93000
4,L000005,C00005,Mortgage,27,45900,Permanent employment,1592000
5,L000006,C00006,Mortgage,33,29900,Self-employed,1138000
6,L000007,C00006,Car Loan,33,29900,Self-employed,135000
7,L000008,C00007,Credit Card,20,57100,Self-employed,21000
8,L000009,C00008,Car Loan,67,24000,Retired,173000
9,L000010,C00009,Mortgage,48,42600,Permanent employment,1570000


In [16]:
loans_df.groupby("loan_type")["original_amount"].agg(
    ["count", "mean", "min", "max"]
).round(0)

,count,mean,min,max
loan_type,,,,
Car Loan,2459,209220.0,50000,700000
Credit Card,4486,28339.0,5000,100000
Mortgage,5084,1545903.0,300000,4022000
Personal Loan,3050,156811.0,10000,500000


In [17]:
current_balances = []

for index, loan in loans_df.iterrows():

    loan_type = loan["loan_type"]
    original_amount = loan["original_amount"]

    if loan_type == "Mortgage":

        remaining_share = np.random.uniform(0.45, 0.95)
        current_balance = original_amount * remaining_share

    elif loan_type == "Personal Loan":

        remaining_share = np.random.uniform(0.20, 0.90)
        current_balance = original_amount * remaining_share

    elif loan_type == "Car Loan":

        remaining_share = np.random.uniform(0.25, 0.90)
        current_balance = original_amount * remaining_share

    else:  # Credit Card

        utilization_rate = np.random.uniform(0.05, 0.85)
        current_balance = original_amount * utilization_rate

    current_balance = round(current_balance / 100) * 100

    current_balances.append(int(current_balance))

In [18]:
loans_df["current_balance"] = current_balances

loans_df.head(10)

,loan_id,customer_id,loan_type,age,monthly_income,employment_status,original_amount,current_balance
0,L000001,C00001,Personal Loan,41,26400,Temporary/Probation/Freelance,186000,106200
1,L000002,C00002,Mortgage,68,18900,Retired,470000,265600
2,L000003,C00003,Mortgage,56,30800,Permanent employment,917000,824100
3,L000004,C00004,Car Loan,49,35700,Permanent employment,93000,71800
4,L000005,C00005,Mortgage,27,45900,Permanent employment,1592000,982400
5,L000006,C00006,Mortgage,33,29900,Self-employed,1138000,667100
6,L000007,C00006,Car Loan,33,29900,Self-employed,135000,72500
7,L000008,C00007,Credit Card,20,57100,Self-employed,21000,15500
8,L000009,C00008,Car Loan,67,24000,Retired,173000,84200
9,L000010,C00009,Mortgage,48,42600,Permanent employment,1570000,1024700


In [19]:
loans_df.groupby("loan_type")["current_balance"].agg(
    ["count", "mean", "min", "max"]
).round(0)

,count,mean,min,max
loan_type,,,,
Car Loan,2459,120228.0,12700,567400
Credit Card,4486,12864.0,300,77900
Mortgage,5084,1079343.0,159400,3400400
Personal Loan,3050,87050.0,2700,388200


In [20]:
interest_rates = []

for index, loan in loans_df.iterrows():

    loan_type = loan["loan_type"]
    employment_status = loan["employment_status"]
    monthly_income = loan["monthly_income"]

    # Grundränta beroende på lånetyp
    if loan_type == "Mortgage":
        interest_rate = np.random.normal(2.9, 0.35)
        interest_rate = np.clip(interest_rate, 2.2, 4.2)

    elif loan_type == "Car Loan":
        interest_rate = np.random.normal(6.5, 1.3)
        interest_rate = np.clip(interest_rate, 3.5, 10.5)

    elif loan_type == "Personal Loan":
        interest_rate = np.random.normal(8.5, 2.0)
        interest_rate = np.clip(interest_rate, 4.5, 15.0)

    else:  # Credit Card
        interest_rate = np.random.normal(17.0, 2.5)
        interest_rate = np.clip(interest_rate, 10.0, 22.0)

    # Riskpåslag för mindre stabil sysselsättning
    if employment_status == "Temporary/Probation/Freelance":
        interest_rate += 0.5

    elif employment_status == "Self-employed":
        interest_rate += 0.3

    elif employment_status == "Unemployed":
        interest_rate += 1.5

    elif employment_status == "Student":
        interest_rate += 1.0

    # Ett litet påslag vid mycket låg inkomst
    if monthly_income < 20000:
        interest_rate += 0.5

    interest_rate = round(interest_rate, 2)

    interest_rates.append(interest_rate)

In [21]:
loans_df["interest_rate"] = interest_rates

loans_df.groupby("loan_type")["interest_rate"].agg(
    ["count", "mean", "min", "max"]
).round(2)

,count,mean,min,max
loan_type,,,,
Car Loan,2459,6.87,3.5,12.50
Credit Card,4486,17.41,10.0,24.00
Mortgage,5084,2.98,2.2,5.56
Personal Loan,3050,8.80,4.5,16.82


In [26]:
remaining_term_months = []

for index, loan in loans_df.iterrows():

    loan_type = loan["loan_type"]
    original_amount = loan["original_amount"]
    current_balance = loan["current_balance"]

    # Hur stor del av ursprungslånet finns kvar?
    remaining_share = current_balance / original_amount

    if loan_type == "Mortgage":
        original_term = np.random.randint(240, 361)  # 20–30 år
        months = max(60, round(original_term * remaining_share))

    elif loan_type == "Personal Loan":
        original_term = np.random.randint(24, 121)   # 2–10 år
        months = max(6, round(original_term * remaining_share))

    elif loan_type == "Car Loan":
        original_term = np.random.randint(24, 85)    # 2–7 år
        months = max(6, round(original_term * remaining_share))

    else:
        # Kreditkort hanteras separat
        months = None

    remaining_term_months.append(months)

In [27]:
monthly_payments = []

for index, loan in loans_df.iterrows():

    loan_type = loan["loan_type"]
    balance = loan["current_balance"]
    annual_interest_rate = loan["interest_rate"] / 100

    if loan_type == "Credit Card":

        credit_limit = loan["original_amount"]

        # Förenklad kreditkortsregel:
        # cirka 5 % av kreditbeloppet
        monthly_payment = credit_limit * 0.05

        # Betalningen ska aldrig vara större än skulden
        monthly_payment = min(monthly_payment, balance)

    else:

        monthly_interest_rate = annual_interest_rate / 12
        months = remaining_term_months[index]

        monthly_payment = (
            balance
            * monthly_interest_rate
            * (1 + monthly_interest_rate) ** months
            / ((1 + monthly_interest_rate) ** months - 1)
        )

    monthly_payment = round(monthly_payment / 10) * 10

    monthly_payments.append(int(monthly_payment))

In [29]:
loans_df["monthly_payment"] = monthly_payments

In [30]:
loans_df.groupby("loan_type")["monthly_payment"].agg(
    ["count", "mean", "min", "max"]
).round(0)

,count,mean,min,max
loan_type,,,,
Car Loan,2459,4726.0,650,24250
Credit Card,4486,1417.0,250,5000
Mortgage,5084,6692.0,1370,19670
Personal Loan,3050,3007.0,110,17020


In [31]:
loans_df["total_monthly_payment"] = (
    loans_df.groupby("customer_id")["monthly_payment"]
    .transform("sum")
)

loans_df["payment_to_income_ratio"] = (
    loans_df["total_monthly_payment"]
    / loans_df["monthly_income"]
)

loans_df[
    [
        "customer_id",
        "monthly_income",
        "total_monthly_payment",
        "payment_to_income_ratio"
    ]
].head(10)

,customer_id,monthly_income,total_monthly_payment,payment_to_income_ratio
0,C00001,26400,5980,0.226515
1,C00002,18900,1870,0.098942
2,C00003,30800,4150,0.134740
3,C00004,35700,3800,0.106443
4,C00005,45900,6640,0.144662
5,C00006,29900,8410,0.281271
6,C00006,29900,8410,0.281271
7,C00007,57100,1050,0.018389
8,C00008,24000,4260,0.177500
9,C00009,42600,6880,0.161502


In [32]:
late_payments = []

for index, loan in loans_df.iterrows():

    payment_ratio = loan["payment_to_income_ratio"]
    employment_status = loan["employment_status"]
    loan_type = loan["loan_type"]

    # Grundsannolikhet för antal sena betalningar
    probabilities = np.array([
        0.86,  # 0 sena betalningar
        0.09,  # 1
        0.03,  # 2
        0.015, # 3
        0.005  # 4
    ])

    # Hög betalningsbörda ökar risken
    if payment_ratio >= 0.40:
        probabilities = np.array([
            0.55,
            0.20,
            0.12,
            0.08,
            0.05
        ])

    elif payment_ratio >= 0.30:
        probabilities = np.array([
            0.70,
            0.16,
            0.07,
            0.045,
            0.025
        ])

    # Sysselsättning påverkar också
    if employment_status == "Unemployed":
        probabilities[0] *= 0.70
        probabilities[2:] *= 1.50

    elif employment_status == "Student":
        probabilities[0] *= 0.85
        probabilities[1:] *= 1.20

    elif employment_status == "Temporary/Probation/Freelance":
        probabilities[0] *= 0.90
        probabilities[1:] *= 1.10

    # Kreditkort och privatlån får lite högre risk
    if loan_type == "Credit Card":
        probabilities[0] *= 0.90
        probabilities[1:] *= 1.10

    elif loan_type == "Personal Loan":
        probabilities[0] *= 0.95
        probabilities[1:] *= 1.05

    # Gör om så att summan blir 100 %
    probabilities = probabilities / probabilities.sum()

    late_payment_count = np.random.choice(
        [0, 1, 2, 3, 4],
        p=probabilities
    )

    late_payments.append(late_payment_count)

In [33]:
loans_df["late_payments"] = late_payments

print(loans_df["late_payments"].value_counts().sort_index())

print("\nAndelar:")
print(
    loans_df["late_payments"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(1)
)

late_payments
0    12013
1     1767
2      709
3      397
4      193
Name: count, dtype: int64

Andelar:
late_payments
0    79.7
1    11.7
2     4.7
3     2.6
4     1.3
Name: proportion, dtype: float64


In [34]:
default_flags = []

for index, loan in loans_df.iterrows():

    late_payment_count = loan["late_payments"]
    payment_ratio = loan["payment_to_income_ratio"]
    employment_status = loan["employment_status"]

    # Grundrisk
    default_probability = 0.005

    # Fler sena betalningar ökar risken
    if late_payment_count == 1:
        default_probability += 0.01

    elif late_payment_count == 2:
        default_probability += 0.04

    elif late_payment_count == 3:
        default_probability += 0.10

    elif late_payment_count >= 4:
        default_probability += 0.22

    # Hög betalningsbörda ökar risken
    if payment_ratio >= 0.40:
        default_probability += 0.08

    elif payment_ratio >= 0.30:
        default_probability += 0.03

    # Sysselsättning påverkar
    if employment_status == "Unemployed":
        default_probability += 0.06

    elif employment_status == "Temporary/Probation/Freelance":
        default_probability += 0.02

    elif employment_status == "Student":
        default_probability += 0.02

    # Håll sannolikheten inom rimliga gränser
    default_probability = min(default_probability, 0.60)

    default_flag = np.random.choice(
        [0, 1],
        p=[
            1 - default_probability,
            default_probability
        ]
    )

    default_flags.append(default_flag)

In [35]:
loans_df["default_flag"] = default_flags

print(loans_df["default_flag"].value_counts())

print("\nDefault rate:")
print(
    round(
        loans_df["default_flag"].mean() * 100,
        2
    ),
    "%"
)

default_flag
0    14663
1      416
Name: count, dtype: int64

Default rate:
2.76 %


In [36]:
default_by_loan_type = loans_df.groupby("loan_type")["default_flag"].agg(
    ["count", "sum", "mean"]
)

default_by_loan_type["default_rate_percent"] = (
    default_by_loan_type["mean"] * 100
).round(2)

default_by_loan_type

,count,sum,mean,default_rate_percent
loan_type,,,,
Car Loan,2459,88,0.035787,3.58
Credit Card,4486,138,0.030762,3.08
Mortgage,5084,118,0.023210,2.32
Personal Loan,3050,72,0.023607,2.36


In [37]:
print("Antal lån:", len(loans_df))

print("\nSaknade värden:")
print(loans_df.isna().sum())

print("\nDuplicerade loan_id:")
print(loans_df["loan_id"].duplicated().sum())

print("\nCurrent balance större än original amount:")
print(
    (loans_df["current_balance"] > loans_df["original_amount"]).sum()
)

print("\nNegativa current balances:")
print(
    (loans_df["current_balance"] < 0).sum()
)

print("\nNegativa monthly payments:")
print(
    (loans_df["monthly_payment"] < 0).sum()
)

Antal lån: 15079

Saknade värden:
loan_id                    0
customer_id                0
loan_type                  0
age                        0
monthly_income             0
employment_status          0
original_amount            0
current_balance            0
interest_rate              0
monthly_payment            0
total_monthly_payment      0
payment_to_income_ratio    0
late_payments              0
default_flag               0
dtype: int64

Duplicerade loan_id:
0

Current balance större än original amount:
0

Negativa current balances:
0

Negativa monthly payments:
0


In [38]:
final_loans_df = loans_df[
    [
        "loan_id",
        "customer_id",
        "loan_type",
        "original_amount",
        "current_balance",
        "interest_rate",
        "monthly_payment",
        "late_payments",
        "default_flag"
    ]
].copy()

final_loans_df.head(10)

,loan_id,customer_id,loan_type,original_amount,current_balance,interest_rate,monthly_payment,late_payments,default_flag
0,L000001,C00001,Personal Loan,186000,106200,8.28,5980,0,0
1,L000002,C00002,Mortgage,470000,265600,2.92,1870,0,0
2,L000003,C00003,Mortgage,917000,824100,2.42,4150,0,0
3,L000004,C00004,Car Loan,93000,71800,6.70,3800,0,0
4,L000005,C00005,Mortgage,1592000,982400,3.50,6640,0,0
5,L000006,C00006,Mortgage,1138000,667100,3.14,5240,1,0
6,L000007,C00006,Car Loan,135000,72500,8.46,3170,0,0
7,L000008,C00007,Credit Card,21000,15500,15.50,1050,0,0
8,L000009,C00008,Car Loan,173000,84200,6.70,4260,0,0
9,L000010,C00009,Mortgage,1570000,1024700,2.67,6880,0,0


In [39]:
print("Antal lån:", len(final_loans_df))

print("\nKolumner:")
print(final_loans_df.columns.tolist())

print("\nSaknade värden:")
print(final_loans_df.isna().sum())

print("\nDuplicerade loan_id:")
print(final_loans_df["loan_id"].duplicated().sum())

Antal lån: 15079

Kolumner:
['loan_id', 'customer_id', 'loan_type', 'original_amount', 'current_balance', 'interest_rate', 'monthly_payment', 'late_payments', 'default_flag']

Saknade värden:
loan_id            0
customer_id        0
loan_type          0
original_amount    0
current_balance    0
interest_rate      0
monthly_payment    0
late_payments      0
default_flag       0
dtype: int64

Duplicerade loan_id:
0


In [40]:
loans_path = project_root / "data" / "loans.csv"

final_loans_df.to_csv(
    loans_path,
    index=False,
    encoding="utf-8-sig"
)

print("CSV sparad här:")
print(loans_path)

CSV sparad här:
c:\Users\ELIYA\Documents\nordic-credit-risk\data\loans.csv
